# EV Purchase Prediction

This is my first Kaggle data competition.

The goal is to predict whether a person will buy an electric vehicle. The competition uses ROC-AUC, so the model needs to give a probability for each person rather than just predicting Yes or No.

Before building a model, I want to understand the data properly and see which features might be useful.

## 1. Load the Data

I'll start by loading the training data, test data, and sample submission file.

The training data contains the target column, `Will_Buy_EV`. The test data does not have this column because those are the rows we need to make predictions for.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')

TARGET = 'Will_Buy_EV'

print('Libraries loaded successfully.')

## 2. First Look at the Data

First, I'll check the size of the datasets, the columns, and a few rows.

This gives me a basic idea of what I'm working with before I start making any changes.

In [ ]:
train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')
sample_submission = pd.read_csv('../data/sample_submission.csv')

print(f'Train shape: {train.shape}')
print(f'Test shape: {test.shape}')
print(f'Sample submission shape: {sample_submission.shape}')

## 3. Missing Values

I'll check for missing values in both the training and test data.

If there are any, we'll need to deal with them before training the model.

In [ ]:
display(train.head())
display(train.tail())

print('Train columns:')
print(train.columns.tolist())

In [ ]:
print('Train info:')
train.info()

## 4. Duplicate Rows

Next, I'll check for duplicate rows.

Duplicate rows are not always a problem, but it is useful to know whether they exist before moving on.

In [ ]:
missing_train = train.isna().sum().sort_values(ascending=False)
missing_test = test.isna().sum().sort_values(ascending=False)

print('Missing values in train:')
display(missing_train[missing_train > 0])

print('Missing values in test:')
display(missing_test[missing_test > 0])

print(f'Total missing values in train: {train.isna().sum().sum()}')
print(f'Total missing values in test: {test.isna().sum().sum()}')

## 5. Target Variable

The target is `Will_Buy_EV`.

It has two possible values:

- `Yes`, the person buys an EV
- `No`, the person does not buy an EV

I'll check the number of people in each group because the balance between the two classes can affect how we evaluate the model.

In [ ]:
print(f'Duplicate rows in train: {train.duplicated().sum():,}')
print(f'Duplicate rows in test: {test.duplicated().sum():,}')

## 6. Feature Types

Now I'll separate the features into numerical and categorical columns.

Some columns contain numbers, such as income and commute distance. Others contain categories, such as gender, city type, and current car type.

Knowing the column types will help when we build the preprocessing pipeline.

In [ ]:
target_counts = train[TARGET].value_counts()
target_percent = train[TARGET].value_counts(normalize=True) * 100

target_summary = pd.DataFrame({
    'Count': target_counts,
    'Percentage': target_percent
})

display(target_summary)

plt.figure(figsize=(7, 5))
sns.countplot(data=train, x=TARGET)
plt.title('Target Distribution')
plt.xlabel('Will Buy EV')
plt.ylabel('Number of Rows')
plt.show()

## 7. Numerical Features

I'll look at the main statistics for the numerical columns, including the minimum, maximum, average, and quartiles.

This can help us spot unusual values and get a feel for the ranges of the features.

## 8. Categorical Features

For the categorical columns, I'll look at how common each category is.

This is useful because some categories may have very few rows, while others may make up most of the dataset.

In [ ]:
numeric_features = train.select_dtypes(include=np.number).columns.tolist()
categorical_features = train.select_dtypes(include=['object', 'string', 'category', 'bool']).columns.tolist()

print('Numerical features:')
print(numeric_features)

print('Categorical features:')
print(categorical_features)

## 9. Low-Cardinality Numerical Features

A few numerical columns only have a small number of possible values.

For example, `Number_of_Cars_Owned` only ranges from 1 to 4, and `Environmental_Concern_Level` ranges from 1 to 5.

I'll check these values directly to see how they are being used in the dataset.

In [ ]:
display(train[numeric_features].describe().T)

## 10. Numerical Features vs EV Purchases

Now I'll compare the numerical features between people who said Yes and people who said No.

The goal here is not to prove that one feature causes someone to buy an EV. I'm just looking for differences that could be useful for a model.

In [ ]:
for col in categorical_features:
    print(f'--- {col} ---')
    display(train[col].value_counts().to_frame('Count'))

## 11. Average Values by Target

I'll calculate the average value of each numerical feature for the two target groups.

This gives me a quick way to compare people who buy an EV with people who do not.

In [ ]:
low_cardinality_numeric = [
    'Number_of_Cars_Owned',
    'Charging_Stations_Near_Home',
    'Charging_Stations_Near_Work',
    'Environmental_Concern_Level'
]

for col in low_cardinality_numeric:
    print(f'{col}: {sorted(train[col].unique().tolist())}')

## 12. Categorical Features vs EV Purchases

Next, I'll check the EV purchase rate for each category.

For example, we can compare the purchase rate for different city types or different car types.

This should give us a better idea of whether some categories are more common among EV buyers.

In [ ]:
numeric_for_plots = [c for c in numeric_features if c != 'id']

for col in numeric_for_plots:
    plt.figure(figsize=(8, 5))
    sns.boxplot(data=train, x=TARGET, y=col)
    plt.title(f'{col} vs {TARGET}')
    plt.show()

## 13. Correlation

I'll also check the correlation between the numerical features and the target.

The target is originally stored as `Yes` and `No`, so I'll temporarily convert it to 1 and 0 for this calculation.

Correlation is only a quick check here. A low correlation does not automatically mean a feature is useless, especially when the relationship is not linear.

In [ ]:
grouped_means = train.groupby(TARGET)[numeric_features].mean().T
display(grouped_means)

## 14. Correlation Heatmap

The heatmap makes the correlations easier to compare.

I'm mainly looking for strong relationships between features and also for features that are very similar to each other.

In [ ]:
for col in categorical_features:
    if col == TARGET:
        continue
    rate_table = (
        train.groupby(col)[TARGET]
        .apply(lambda x: (x == 'Yes').mean())
        .sort_values(ascending=False)
        .to_frame('EV_Purchase_Rate')
    )
    print(f'--- {col} ---')
    display(rate_table)

    plt.figure(figsize=(8, 5))
    sns.barplot(x=rate_table.index, y=rate_table['EV_Purchase_Rate'])
    plt.title(f'EV Purchase Rate by {col}')
    plt.xlabel(col)
    plt.ylabel('Purchase Rate')
    plt.xticks(rotation=30)
    plt.show()

## 15. Checking the ID Column

The `id` column is just an identifier, so I don't expect it to contain useful information about whether someone buys an EV.

I'll still check it briefly before deciding whether to leave it out of the model.

In [ ]:
target_numeric = train[TARGET].map({'No': 0, 'Yes': 1})

correlation_data = train[numeric_features].copy()
correlation_data[TARGET] = target_numeric

correlation_matrix = correlation_data.corr(numeric_only=True)

target_correlations = correlation_matrix[TARGET].drop(TARGET).sort_values(key=abs, ascending=False)
display(target_correlations.to_frame('Correlation with Will_Buy_EV'))

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Numerical Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## 16. What I Found

A few things stand out from the first look at the data:

- The target is imbalanced, with fewer people saying Yes than No.
- Income looks noticeably different between the two target groups.
- Environmental concern also looks quite different between the groups.
- Some categorical features may have useful differences in EV purchase rates.
- There are no missing values to deal with.
- There are no duplicate rows in the training or test data.
- The `id` column looks like an identifier rather than a useful feature.

These are early observations. I'll test whether these patterns actually help when we train models.

In [ ]:
print(f'Unique train IDs: {train.id.nunique():,} / {len(train):,}')
print(f'Unique test IDs: {test.id.nunique():,} / {len(test):,}')
print(f'Train ID range: {train.id.min()} to {train.id.max()}')
print(f'Test ID range: {test.id.min()} to {test.id.max()}')

## 17. What's Next

The next step is to prepare the data for modelling.

I'll handle the categorical columns, keep the numerical columns in a suitable format, split the training data for validation, and then build a simple baseline model.

After that, I'll compare stronger models and see what improves the ROC-AUC score.

## 18. Competition Metric

The competition uses ROC-AUC to evaluate the predictions.

This means the model should produce a probability of buying an EV for each person. I'll use the validation data to compare models before making submissions to Kaggle.